# AiXbio Hackathon â€” ESM-2 Toxin Screener Demo

This interactive notebook demonstrates our 88% accurate, zero-shot toxin screening tool against ProteinMPNN redesigns.
It uses a mechanistic interpretability approach (Direct Logit Attribution & Sparse Autoencoders) to explain *why* a sequence is flagged.

In [ ]:
# ── 0. Install Dependencies ──────────────────────────────────────────────
# Run this cell once to install required packages
!pip install -q torch transformers biopython scikit-learn matplotlib numpy


In [ ]:
# â”€â”€ 1. Setup and Load Model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch, json, numpy as np, matplotlib.pyplot as plt
from screen import load_model, score_seq, risk_level, explain, FEATURE_NAMES

print("Loading ESM-2 and Toxin Circuit Probe...")
tok, esm2, probe, probe_dir_np = load_model()
print("Ready.")

### Test a Sequence
Try pasting a known toxin, a random protein, or an AI redesign below.

In [ ]:
# â”€â”€ 2. Run Toxin Screen â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

TEST_SEQUENCE = "MKTIIALSYIFCLVFADYKDDDDKGAPSITCPPCGVKRNDGCCPIVCPPGGCPKCPRC"
# (This is a sample toxin snippet. Replace it with any sequence!)

score = score_seq(TEST_SEQUENCE, tok, esm2, probe)
level = risk_level(score)

print(f"Sequence Length: {len(TEST_SEQUENCE)} AA")
print(f"Toxin Probability: {score:.4f}")
print(f"Risk Level: {level}")
if score > 0.5:
    print("ðŸš¨ WARNING: High probability of toxic function. Flagged for review.")
else:
    print("âœ… SAFE: No toxic function detected.")

### Mechanistic Explanation
How did the model make this decision? We use Direct Logit Attribution (DLA) to see which layers contributed to the final score.

In [ ]:
# â”€â”€ 3. Explain Decision (Layer Attribution) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

explanation = explain(TEST_SEQUENCE, tok, esm2, probe_dir_np, list(FEATURE_NAMES.keys()))
dpa = explanation['layer_attribution']
top_layers = explanation['top_discriminating_layers']

plt.figure(figsize=(10, 4))
colors = ['#E07B39' if val > 0 else '#2E86AB' for val in dpa]
plt.bar(range(1, 34), dpa, color=colors)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Direct Logit Attribution: Which layers drove the decision?')
plt.xlabel('ESM-2 Layer')
plt.ylabel('Attribution Score')
plt.show()

print(f"Key layers driving this decision: {top_layers}")